# SettleTax Transaction Classifier — Test Notebook

Upload a real bank statement PDF and run the 4-layer classifier on every transaction.

**Layers:**
1. **Structural** — Self-transfers, bank charges, reversals, ATM
2. **User History** — Previously categorised counterparties
3. **Rule Engine** — 150+ Nigerian-specific keyword patterns
4. **LLM Fallback** — Claude/GPT for ambiguous transactions

## Step 1: Install Dependencies

In [1]:
!pip install -q pikepdf pdfplumber pandas openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 76.9 MB/s eta 0:00:00


## Step 1b: Upload Classifier Module

Upload `settletax_classifier.py` from the `classifier/` folder. This file contains the 4-layer classification engine.

In [ ]:
import sys, os

# Upload settletax_classifier.py to Colab
try:
    from google.colab import files
    if not os.path.exists('settletax_classifier.py'):
        print('Upload settletax_classifier.py from the classifier/ folder:')
        uploaded = files.upload()
        if 'settletax_classifier.py' not in uploaded:
            raise FileNotFoundError('settletax_classifier.py was not uploaded.')
    else:
        print('settletax_classifier.py already exists.')
except ImportError:
    # Not running in Colab — assume file is in the current directory or parent
    if not os.path.exists('settletax_classifier.py'):
        # Try parent directory (if running from tests/ or similar)
        parent = os.path.join(os.path.dirname(os.getcwd()), 'settletax_classifier.py')
        if os.path.exists(parent):
            sys.path.insert(0, os.path.dirname(parent))
        else:
            print('Warning: settletax_classifier.py not found. Make sure it is in the working directory.')

sys.path.insert(0, '.')

# Verify import works
from settletax_classifier import BankStatementParser, SettleTaxClassifier
print('Classifier module loaded successfully.')

Upload settletax_classifier.py from the classifier/ folder:


Saving settletax_classifier.py to settletax_classifier.py
Classifier module loaded successfully.


## Step 2: Upload & Unlock Bank Statement PDF

In [ ]:
import os
from getpass import getpass
from pikepdf import Pdf, PasswordError

# Upload the PDF
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_filename = next(iter(uploaded))
except ImportError:
    pdf_filename = input('Enter path to PDF file: ')

print(f'PDF: {pdf_filename}')

def unlock_if_needed(input_path: str) -> str:
    try:
        with Pdf.open(input_path):
            print('PDF is not encrypted.')
            return input_path
    except PasswordError:
        print('PDF is encrypted.')
        password = getpass('Enter PDF password: ')
        with Pdf.open(input_path, password=password) as pdf:
            base, ext = os.path.splitext(input_path)
            unlocked_path = base + '_unlocked' + ext
            pdf.save(unlocked_path)
            print(f'Unlocked: {unlocked_path}')
            return unlocked_path

pdf_path = unlock_if_needed(pdf_filename)

Saving 8540708018-26090142885 2.pdf to 8540708018-26090142885 2.pdf
PDF: 8540708018-26090142885 2.pdf
PDF is not encrypted.


## Step 3: Extract Transactions from PDF

In [ ]:
# BankStatementParser was imported in Step 1b
parser = BankStatementParser()
tx_df = parser.parse(pdf_path)

print(f'Extracted {len(tx_df)} transactions')
print(f'Columns: {tx_df.columns.tolist()}')
print(f'Debits: {(tx_df.direction=="debit").sum()}, Credits: {(tx_df.direction=="credit").sum()}')
tx_df[['narration', 'amount', 'direction']].head(10)

Extracted 255 transactions
Columns: ['Trans_Date', 'Reference', 'Remarks', 'Value_Date', 'Credits', 'Debits', 'Balance', 'direction', 'amount', 'narration', 'date']
Debits: 227, Credits: 28


,narration,amount,direction
0,,0.00,debit
1,App To Paycomopay RERELOLUWA OBAFEMIMOSES,30000.00,debit
2,TRANSACTION CHARGE-App To Paycomopay RERELOLUWA O,26.88,debit
3,TRANSACTION CHARGE-App To Opay Digital Service...,26.88,debit
4,App To Opay Digital Services Limited AFEEZ TAI,38000.00,debit
5,Appmeat To Opay Digital Services Limited BABAT,26000.00,debit
6,TRANSACTION CHARGE-Appmeat To Opay Digital Ser...,26.88,debit
7,TRANSACTION CHARGE-App To MONIEPOINT MICROFINANCE,26.88,debit
8,App To MONIEPOINT MICROFINANCE BANK POS Transf,15500.00,debit
9,TRANSACTION CHARGE-App To Opay Digital Service...,26.88,debit


## Step 4: Clean & Prepare Data

In [ ]:
# BankStatementParser already handles column normalization, amount parsing,
# direction detection, and narration extraction.
# This cell just verifies the data and does any final cleanup.

# Remove rows with zero/NaN amounts (PDF artifacts like page breaks)
tx_df = tx_df[tx_df['amount'] > 0].reset_index(drop=True)

# Show summary
print(f'Clean transactions: {len(tx_df)}')
print(f'Debits: {(tx_df.direction=="debit").sum()}, Credits: {(tx_df.direction=="credit").sum()}')
print(f'Total debits:  ₦{tx_df[tx_df.direction=="debit"]["amount"].sum():,.2f}')
print(f'Total credits: ₦{tx_df[tx_df.direction=="credit"]["amount"].sum():,.2f}')
tx_df[['narration', 'amount', 'direction', 'date']].head(10)

Clean transactions: 252
Debits: 224, Credits: 28
Total debits:  ₦10,453,868.64
Total credits: ₦10,368,750.00


,narration,amount,direction,date
0,App To Paycomopay RERELOLUWA OBAFEMIMOSES,30000.00,debit,28-Sep-2025
1,TRANSACTION CHARGE-App To Paycomopay RERELOLUWA O,26.88,debit,28-Sep-2025
2,TRANSACTION CHARGE-App To Opay Digital Service...,26.88,debit,28-Sep-2025
3,App To Opay Digital Services Limited AFEEZ TAI,38000.00,debit,28-Sep-2025
4,Appmeat To Opay Digital Services Limited BABAT,26000.00,debit,28-Sep-2025
5,TRANSACTION CHARGE-Appmeat To Opay Digital Ser...,26.88,debit,28-Sep-2025
6,TRANSACTION CHARGE-App To MONIEPOINT MICROFINANCE,26.88,debit,28-Sep-2025
7,App To MONIEPOINT MICROFINANCE BANK POS Transf,15500.00,debit,28-Sep-2025
8,TRANSACTION CHARGE-App To Opay Digital Service...,26.88,debit,28-Sep-2025
9,App To Opay Digital Services Limited OLAIDE AD,10000.00,debit,28-Sep-2025


## Step 5: Configure Classifier

**Edit the account name below to match the bank statement.**

In [ ]:
# SettleTaxClassifier was imported in Step 1b

# ══════════════════════════════════════════════════
# EDIT THESE VALUES TO MATCH YOUR BANK STATEMENT
# ══════════════════════════════════════════════════
ACCOUNT_NAME = 'YOUR NAME HERE'     # e.g., 'OBAFEMI-MOSES MOSINMILOLUWA'
BUSINESS_NAME = 'YOUR BUSINESS'     # e.g., 'SETTLE TAX COMPANY LTD' (optional)

# LLM Configuration — set your API key to enable Layer 4 classification
# Anthropic (Claude): get key from https://console.anthropic.com/
# OpenAI (GPT):       get key from https://platform.openai.com/api-keys
LLM_API_KEY = ''                  # e.g., 'sk-ant-...' or 'sk-...'
LLM_PROVIDER = 'anthropic'          # 'anthropic' or 'openai'
LLM_MODEL = 'claude-sonnet-4-20250514'  # or 'gpt-4o' for OpenAI
# ══════════════════════════════════════════════════

classifier = SettleTaxClassifier(
    account_name=ACCOUNT_NAME,
    account_names=[BUSINESS_NAME] if BUSINESS_NAME != 'YOUR BUSINESS' else None,
    llm_api_key=LLM_API_KEY,
    llm_provider=LLM_PROVIDER,
    llm_model=LLM_MODEL,
)

llm_status = 'ENABLED' if LLM_API_KEY else 'DISABLED (set LLM_API_KEY above)'
print(f'Classifier ready. Account: {ACCOUNT_NAME}')
print(f'LLM Layer 4: {llm_status}')

Classifier ready. Account: YOUR NAME HERE
LLM Layer 4: ENABLED


## Step 6: Run Classification

In [ ]:
# Classify all transactions
# llm_enabled=True sends unclassified transactions to LLM (Layer 4)
# Requires LLM_API_KEY to be set in Step 5. If no key, LLM returns unclassified.
results_df = classifier.classify_batch(tx_df, llm_enabled=True)

# Print stats
classifier.print_stats()

# Show results
results_df[['narration', 'amount', 'direction', 'st_category', 'st_source',
            'st_confidence', 'st_needs_review', 'st_counterparty']].head(20)


SettleTax Classification Results
Total transactions: 252
  Layer 1 (Structural):       3 (1.2%)
  Layer 2 (User History):     0 (0.0%)
  Layer 3 (Rules):           15 (6.0%)
  Layer 4 (LLM):            234 (92.9%)
  Unclassified:               0 (0.0%)


,narration,amount,direction,st_category,st_source,st_confidence,st_needs_review,st_counterparty
0,App To Paycomopay RERELOLUWA OBAFEMIMOSES,30000.00,debit,Transfer,llm,0.80,False,None
1,TRANSACTION CHARGE-App To Paycomopay RERELOLUWA O,26.88,debit,Bank Charges,llm,0.95,False,None
2,TRANSACTION CHARGE-App To Opay Digital Service...,26.88,debit,Bank Charges,llm,0.95,False,None
3,App To Opay Digital Services Limited AFEEZ TAI,38000.00,debit,Transfer,llm,0.80,False,None
4,Appmeat To Opay Digital Services Limited BABAT,26000.00,debit,Transfer,llm,0.80,False,None
5,TRANSACTION CHARGE-Appmeat To Opay Digital Ser...,26.88,debit,Bank Charges,llm,0.95,False,None
6,TRANSACTION CHARGE-App To MONIEPOINT MICROFINANCE,26.88,debit,Bank Charges,llm,0.95,False,None
7,App To MONIEPOINT MICROFINANCE BANK POS Transf,15500.00,debit,Transfer,llm,0.85,False,None
8,TRANSACTION CHARGE-App To Opay Digital Service...,26.88,debit,Bank Charges,llm,0.95,False,None
9,App To Opay Digital Services Limited OLAIDE AD,10000.00,debit,Transfer,llm,0.80,False,None


## Step 7: Results by Category

In [ ]:
# Breakdown by category
category_summary = (
    results_df
    .groupby(['st_category', 'st_type'])
    .agg(
        count=('amount', 'count'),
        total=('amount', 'sum'),
        avg_confidence=('st_confidence', 'mean'),
    )
    .sort_values('total', ascending=False)
)

print('\nClassification Breakdown:')
print('=' * 80)
category_summary


Classification Breakdown:


count       total  avg_confidence
st_category             st_type                                   
Transfer                expense     61  8895861.75        0.804098
                        income      11  6193750.00        0.850000
Income                  income       7  3235000.00        0.814286
Capital Gains Cost      expense      1   500000.00        0.900000
Drawings                expense     19   401625.00        0.747368
Utilities               expense      7   300576.25        0.842857
Fuel                    income       3   290000.00        0.883333
Insurance               expense      2   276500.00        0.900000
Training                expense      2   260000.00        0.800000
Consulting Income       income       2   240000.00        0.875000
Donations               expense     14    66000.00        0.882143
Utilities               income       1    50000.00        0.800000
Repairs and Maintenance expense      3    46000.00        0.833333
Salaries and Wages      expense      1    30000.00        0.800000
Fuel                    expense      2    20026.88        0.780000
Telephone               expense      2    11000.00        0.925000
Bank Charges            expense    113     4278.76        0.968319
Travel Expenses         expense      1     2000.00        0.900000

## Step 8: Classification Source Distribution

In [ ]:
# Breakdown by classification source
source_summary = (
    results_df
    .groupby('st_source')
    .agg(
        count=('amount', 'count'),
        avg_confidence=('st_confidence', 'mean'),
        needs_review=('st_needs_review', 'sum'),
    )
)

print('\nSource Distribution:')
print('=' * 50)
source_summary


Source Distribution:


,count,avg_confidence,needs_review
st_source,,,
llm,234,0.892308,12
rule,15,0.754000,9
structural,3,0.990000,0


## Step 9: Low Confidence — Needs Review

In [ ]:
# Transactions that need user review
review_df = results_df[results_df['st_needs_review'] == True]

print(f'\n{len(review_df)} transactions need review:')
print('=' * 80)

if len(review_df) > 0:
    review_df[['narration', 'amount', 'direction', 'st_category',
               'st_source', 'st_confidence', 'st_explanation']].head(30)
else:
    print('All transactions classified with high confidence!')


21 transactions need review:


## Step 10: Unclassified Transactions

In [ ]:
# Unclassified transactions (need LLM or manual categorisation)
unclass_df = results_df[results_df['st_source'] == 'unclassified']

print(f'\n{len(unclass_df)} unclassified transactions:')
print('=' * 80)

if len(unclass_df) > 0:
    unclass_df[['narration', 'amount', 'direction', 'st_counterparty']].head(30)
else:
    print('All transactions classified!')


0 unclassified transactions:
All transactions classified!


## Step 11: Export Results

In [ ]:
# Export to Excel
output_file = 'classified_statement.xlsx'
results_df.to_excel(output_file, index=False)
print(f'Saved to {output_file}')

# Download in Colab
try:
    from google.colab import files
    files.download(output_file)
except ImportError:
    print(f'File saved locally at: {os.path.abspath(output_file)}')

Saved to classified_statement.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>